# Suntime

Discovering time based on the position of the sun.

[Copyright &copy; Anoduck, The Anonymous Duck; 2025](https://anoduck.mit-license.org)

## Previous Work

### Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
if not os.path.exists("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/results"):
  os.mkdir("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/results")

# Variables for runtime
batch = False

### Setup Environment

In [1]:
import os
import sys

class myenv:

  def __init__(self) -> None:
    self.setup_os()
    self.init_env()
    self.install_sam2()
    self.acquire_adaptershadow()
    self.load_imports()
    self.read_image()

  def setup_os(self):
    import os
    os.chdir("/content")
    global CODE_DIR
    CODE_DIR = "/content/suntime"
    print("Done...")

  def install_sam2(self):
    CWD = os.getcwd()
    ROOT = "/"
    print(f"This is the current dir: {CWD}")
    HOME = os.path.join(ROOT, "content")
    if os.path.exists(HOME):
      print(f"Home exists: {HOME}")
      %cd {HOME}
    else:
      print(f"Home does not exist: {HOME}")
      !mkdir {HOME}
      %cd {HOME}
    print(f"New Current Dir: {os.getcwd()}")
    # Check if GroundingDINO directory already exists before cloning
    if not os.path.exists("sam2"):
      !git clone https://github.com/facebookresearch/sam2
    print(f"Current dir contents: {os.listdir()}")
    %cd ./sam2
    print(f"Current dir contents: {os.listdir()}")
    # Check for requirements.txt before installing
    if os.path.exists("requirements.txt"):
        !pip install -r requirements.txt
        !pip install -q -e .
    %cd /content

  def acquire_adaptershadow(self):
    HOME = "/content"
    print(HOME)
    %cd {HOME}
    # Check if GroundingDINO directory already exists before cloning
    if not os.path.exists("AdapterShadow"):
        !git clone https://github.com/LeipingJie/AdapterShadow.git
    print(f"Current dir contents: {os.listdir()}")

  def init_env(self):
    # !conda info --envs
    # Updating the environment.
    !pip install -U opencv-python matplotlib pytesseract shadowfinder pillow
    !pip install -U git+https://github.com/pingswept/pysolar
    !pip install torch torchvision transformers optimum[exporters,onnxruntime] opencv-python pillow datasets samexporter
    # Redundant installations removed for clarity and efficiency
    # !pip install git+https://github.com/IDEA-Research/GroundingDINO.git
    # !pip install git+https://github.com/facebookresearch/segment-anything.git
    print("The environemnt has beed updated...")

  def load_imports(self):
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    import cv2 as cv
    import torch
    from PIL import Image as PIMAGE # Changed import alias to avoid conflict
    # from optimum.onnxruntime import ORTModelForImageSegmentation
    from transformers import SamProcessor, SamModel # Added SamModel import
    from transformers import pipeline, AutoProcessor, AutoModelForZeroShotObjectDetection, SamProcessor, SamModel
    from datasets import load_dataset
    import tempfile
    import os
    import random as rng
    import numpy as np
    from matplotlib import pyplot as plt
    import google.colab.patches as colab
    from google.colab.patches import cv_imshow as colab_show
    import math
    import glob
    from scipy import ndimage
    import mpl_toolkits.mplot3d.axes3d as p3
    import sys
    import datetime
    import pytz
    import pytesseract
    import shadowfinder
    import onnxruntime as ort # Added onnxruntime import

    print("Done importing modules.")

  def read_image(self):
    import cv2 as cv
    global img # Ensure img is globally accessible
    img = cv.imread('/content/drive/MyDrive/Colab Notebooks/suntime-opencv/IMAG0692_V1xF8JAe.jpg', 1)

In [2]:
def env_reload():
  gcenv = myenv() # Simply instantiate the class

env_reload()

Done...
  Cloning https://github.com/pingswept/pysolar to /tmp/pip-req-build-vqvqldj1
  Running command git clone --filter=blob:none --quiet https://github.com/pingswept/pysolar /tmp/pip-req-build-vqvqldj1
  Resolved https://github.com/pingswept/pysolar to commit 14989d5b8bac33e1357ba021de4b9829b9312e98
  Preparing metadata (setup.py) ... done
The environemnt has beed updated...
This is the current dir: /content
Home exists: /content
/content
New Current Dir: /content
Cloning into 'sam2'...
remote: Enumerating objects: 1070, done.
remote: Total 1070 (delta 0), reused 0 (delta 0), pack-reused 1070 (from 1)
Receiving objects: 100% (1070/1070), 128.11 MiB | 14.48 MiB/s, done.
Resolving deltas: 100% (381/381), done.
Current dir contents: ['.config', 'sam2', 'sample_data']
/content/sam2
Current dir contents: ['CONTRIBUTING.md', 'LICENSE_cctorch', '.watchmanconfig', '.gitignore', '.github', 'setup.py', 'MANIFEST.in', 'backend.Dockerfile', 'sav_dataset', 'LICENSE', 'checkpoints', '.clang-form

In [ ]:
def save_img(cvimg: np.ndarray, label: str) -> bool:
  write_path = os.path.join("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/results", f"{label}.jpg")
  cv.imwrite(write_path, cvimg)
  return True

### Image Utils for Analemma

In [ ]:
class Image:
  @classmethod
  def stackImages(cls, imgArray, scale, lables=None):
      if lables is None:
          lables = []
      sizeW = imgArray[0][0].shape[1]
      sizeH = imgArray[0][0].shape[0]
      rows = len(imgArray)
      cols = len(imgArray[0])
      rowsAvailable = isinstance(imgArray[0], list)
      width = imgArray[0][0].shape[1]
      height = imgArray[0][0].shape[0]
      if rowsAvailable:
          for x in range(0, rows):
              for y in range(0, cols):
                  imgArray[x][y] = cv.resize(imgArray[x][y], (int(sizeW * scale), int(sizeH * scale)))
                  if len(imgArray[x][y].shape) == 2: imgArray[x][y] = cv.cvtColor(imgArray[x][y], cv.COLOR_GRAY2BGR)
          imageBlank = np.zeros((height, width, 3), np.uint8)
          hor = [imageBlank] * rows
          hor_con = [imageBlank] * rows
          for x in range(0, rows):
              hor[x] = np.hstack(imgArray[x])
              hor_con[x] = np.concatenate(imgArray[x])
          try:
              ver = np.vstack(hor)
              ver_con = np.concatenate(hor)
          except:
              pass
      else:
          for x in range(0, rows):
              imgArray[x] = cv.resize(imgArray[x], (int(sizeW * scale), int(sizeH * scale)))
              if len(imgArray[x].shape) == 2: imgArray[x] = cv.cvtColor(imgArray[x], cv.COLOR_GRAY2BGR)
          hor = np.hstack(imgArray)
          hor_con = np.concatenate(imgArray)
          ver = hor
      if len(lables) != 0:
          eachImgWidth = int(ver.shape[1] / cols)
          eachImgHeight = int(ver.shape[0] / rows)
          for d in range(0, rows):
              for c in range(0, cols):
                  cv.rectangle(ver, (c * eachImgWidth, eachImgHeight * d),
                                (c * eachImgWidth + len(lables[d][c]) * 13 + 27, 30 + eachImgHeight * d),
                                (255, 255, 255), cv.FILLED)
                  cv.putText(ver, lables[d][c], (eachImgWidth * c + 10, eachImgHeight * d + 20),
                              cv.FONT_HERSHEY_COMPLEX, 0.7, (255, 0, 255), 2)
      return ver

  @classmethod
  def warp(cls, dst: np.ndarray, transformation_matrix: list) -> np.ndarray:
      w, h = dst.shape[:2]
      return cv.warpPerspective(dst, transformation_matrix, (w, h))

  @classmethod
  def get_formated_canny(cls, image: np.ndarray) -> np.ndarray:
      """
      Formats an image to gray, blur, and lastly to canny

      :param image: Source image
      :return: Canny image
      """
      img = image.copy()
      gray = Image.cvt_to_gray(img)
      blur = cv.GaussianBlur(gray, (5, 5), 1)
      canny = cv.Canny(blur, 10, 50)
      return canny

  @classmethod
  def size_reduction(cls, canvas: np.ndarray, size_reduction: float) -> np.ndarray:
      """
      | Reduces image size by cutting of a percentage of pixels starting from the image outlines

      :param canvas: Source image
      :param size_reduction: Percentage of pixels that gets cut of
      """
      canvas = Image.cvt_to_gray(canvas)
      h, w = canvas.shape[:2]
      reduce_pixels_h = int(((h / 100) * size_reduction) / 2)
      reduce_pixels_w = int(((w / 100) * size_reduction) / 2)

      x = reduce_pixels_w
      w = w - reduce_pixels_w
      y = reduce_pixels_h
      h = h - reduce_pixels_h
      return canvas[y:h, x:w]

  @classmethod
  def cvt_to_gray(cls, image: np.ndarray) -> np.ndarray:
      image = image.copy()
      if len(image.shape) < 3:
          return image

      channels = image.shape[2]
      match channels:
          case 3:
              try:
                  image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
              except:
                  try:
                      image = cv.cvtColor(image, cv.COLOR_RGB2GRAY)
                  except:
                      try:
                          image = cv.cvtColor(image, cv.COLOR_HSV2BGR)
                          image = cv.cvtColor(image, cv.COLOR_RGB2GRAY)
                      except:
                          print("Image format not supported")
                          assert ValueError
      return image

  @classmethod
  def show(cls, img: np.ndarray, winname="test", destroy=False) -> None:
      cv.imshow(winname, img)
      cv.waitKey(99999999)
      if destroy:
          cv.destroyWindow(winname)

### Analemma

In [ ]:
def analemma(image) -> tuple:
  radius = int(51)
  last_center = (0.0, 0.0)
  # reduce image size by 20px on all sides and auto converts it to gray
  (h, w) = image.shape[:2]
  h2 = h - 20
  w2 = w - 20
  image_copy = cv.resize(image, (w2, h2))
  image = Image.size_reduction(image, 20)
  print(f"Image copy shape: {image_copy.shape} and image shape: {image.shape}")
  # insure radius is odd
  print(f"Radius: {radius}")
  if int(radius) % 2:
      pass
  else:
      radius += 1
  # blur the image
  grey = Image.cvt_to_gray(image)
  try:
      blur = cv.GaussianBlur(grey, (int(radius), int(radius)), cv.BORDER_DEFAULT)
  except Exception:
      blur = cv.medianBlur(grey, int(radius))

  # calculate minMax method
  minMaxMethod = image.copy()
  # grey = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
  # method = cv.TM_CCOEFF_NORMED
  # res = cv.matchTemplate(img,template,method)
  (minVal, maxVal, minLoc, maxLoc) = cv.minMaxLoc(blur)
  minMaxCenter = maxLoc
  # apply the minMax method
  cv.circle(minMaxMethod, maxLoc, int(radius), (0, 0, 0), 2)
  # prepare the image for the robust method
  thresh = cv.threshold(blur, 210, 225, cv.THRESH_BINARY)[1]
  erode = cv.erode(thresh, None, iterations=7)
  dilate = cv.dilate(erode, None, iterations=4)
  canny = Image.get_formated_canny(dilate)
  # calculate robust method
  points = np.argwhere(canny > 0)
  robustCenter, radius = cv.minEnclosingCircle(points)
  # apply robust method
  robustMethod = image.copy()
  x = int(robustCenter[1])
  y = int(robustCenter[0])
  rad = int(radius)
  cv.circle(robustMethod, (x, y), rad, (300, 100, 100), 2)
  # debug print
  print("lastCenter: " + str(last_center))
  print("dist: " + str(math.dist(robustCenter, last_center)))
  print("robustCenter: " + str(robustCenter))
  print("maxLoc: " + str(robustCenter))
  print("----------------------------------")
  # determine which method to use
  center = minMaxCenter
  if robustCenter == (0.0, 0.0):
      if last_center != (0.0, 0.0):
          if not (math.dist(minMaxCenter, last_center) < 50):
              center = robustCenter
      else:
          center = (0.0, 0.0)
  # put text and highlight the center
  Cx, Cy = center
  image1 = cv.circle(image_copy, (Cx, Cy), 5, (0, 255, 12), -1)
  image2 = cv.putText(
      image1,
      "centroid",
      (Cx - 25, Cy - 25),
      cv.FONT_HERSHEY_SIMPLEX,
      2,
      (0, 255, 12),
      2,
  )
  print(f"Center: {center}, Radius: {int(radius)}")
  colab_show(image2)
  return image2, center

# img = cropped_img
sunid_img, center = analemma(img)

### Use PyTorch, ONNX, and SAM to detect shadows